In [40]:
import os
import json
import pandas as pd
import traceback

In [41]:
from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()
openai_key=os.getenv("OPENAI_API")

In [42]:
llm2=ChatOpenAI(
    model="gpt-3.5-turbo",
    api_key=openai_key
)
llm = ChatOllama(
    model ="llama3.1:latest",
    temperature=1
)

In [43]:
from langchain_openai import OpenAI
from langchain_core.prompts import PromptTemplate
from langchain_community.callbacks.manager import get_openai_callback
import PyPDF2

In [44]:
from langchain_core.output_parsers import StrOutputParser

In [45]:
Response_json = {
    "1":{
        "mcq":"multiple choice question",
        "options":{
            "a":"choice here",
            "b":"choice here",
            "c":"choice here",
            "d":"choice here",
        },
        "correct":"correct answer",
    },
    "2":{
        "mcq":"multiple choice question",
        "options":{
            "a":"choice here",
            "b":"choice here",
            "c":"choice here",
            "d":"choice here",
        },
        "correct":"correct answer",
    },
    "3":{
        "mcq":"multiple choice question",
        "options":{
            "a":"choice here",
            "b":"choice here",
            "c":"choice here",
            "d":"choice here",
        },
        "correct":"correct answer",
    },
}

In [46]:
template = """
Text:{text}
You are an expert MCQ maker. Given the above text, it is your job to \
create quiz of {number} multiple choice for {subject} students in {tone} tone.
Make sure the questions are not repeated and check all the questions to be conforming the text as well.
Ensure to make {number} MCQs
### Response_json
{response_json}

"""

In [47]:
quiz_prompt = PromptTemplate(
    input_variables=["text","number","subject","tone","response_json"],
    template=template
)

In [48]:
quizreview_chain = quiz_prompt | llm | StrOutputParser()

In [93]:
template2 = """
You are an expert in english grammar and writer.

Given a Multiple Choice Quiz for {subject} students.

You need to:
1. Evaluate quiz complexity.
2. Give complexity analysis in max 50 words.
3. Update questions if needed.
4. Return ONLY valid JSON.

Quiz_mcq:
{quiz}

Return output in this format:

{{
    "complexity_analysis":"...",
    "quiz": {response_json}
}}
"""

In [101]:
quiz_eval_prompt = PromptTemplate(
    input_variables=["subject","quiz","response_json"],
    template=template2
)

In [102]:
quizeval_chain = quiz_eval_prompt | llm

In [103]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import JsonOutputParser

In [115]:
final_chain = ({  "quiz": quizreview_chain, "subject" : lambda x: x["subject"], "response_json": lambda x: json.dumps(Response_json) } | quiz_eval_prompt | llm | JsonOutputParser())

In [116]:
file_path = r"C:\Users\maaz7\mcqgenerator\data.txt"

In [117]:
with open(file_path, 'r') as file:
    text = file.read()

In [118]:
json.dumps(Response_json)

'{"1": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "2": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "3": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}}'

In [119]:
number = 5
subject="Data science"
tone="simple"

In [120]:
#how to check token usage 
# with get_openai_callback() as cb:
#     response=final_chain(
#         {
#             "text":text,
#             "number":number,
#             "subject":subject,
#             "tone":tone,
#             "response_json":json.dumps(Response_json)
#         }
#     )

response = final_chain.invoke({
             "text":text,
             "number":number,
             "subject":subject,
             "tone":tone,
             "response_json":json.dumps(Response_json)
         })

In [121]:
response

{'complexity_analysis': 'Moderate to high complexity, requiring some knowledge of data science and language models.',
 'quiz': {'1': {'mcq': 'What is a Large Language Model (LLM)?',
   'options': {'a': 'A type of neural network that can only analyze text',
    'b': 'A neural network trained on vast amounts of text for natural language processing tasks',
    'c': 'A model that can only generate, but not summarize or translate text',
    'd': 'A type of AI model that can only learn from structured data'},
   'correct': 'b'},
  '2': {'mcq': 'Which of the following is a drawback of LLMs?',
   'options': {'a': 'They are highly parallelizable',
    'b': 'They can be biased or inaccurate if trained on bad data',
    'c': 'They can only generate text, but not summarize or translate it',
    'd': 'They require very little training data'},
   'correct': 'b'},
  '3': {'mcq': 'What is the name of the architecture that LLMs are typically based on?',
   'options': {'a': 'Transformer',
    'b': 'Recu

In [123]:
quiz=response.get("quiz")

In [126]:
quiz_table = []
for key, value in quiz.items():
    mcq=value["mcq"]
    options = " | ".join(
        [
        f"{option} : {option_value}"
        for option, option_value in value["options"].items()
        ]
    )
    correct=value["correct"]
    quiz_table.append({"MCQ": mcq,"Choices" : options, "Correct" : correct})

In [127]:
quiz_table


[{'MCQ': 'What is a Large Language Model (LLM)?',
  'Choices': 'a : A type of neural network that can only analyze text | b : A neural network trained on vast amounts of text for natural language processing tasks | c : A model that can only generate, but not summarize or translate text | d : A type of AI model that can only learn from structured data',
  'Correct': 'b'},
 {'MCQ': 'Which of the following is a drawback of LLMs?',
  'Choices': 'a : They are highly parallelizable | b : They can be biased or inaccurate if trained on bad data | c : They can only generate text, but not summarize or translate it | d : They require very little training data',
  'Correct': 'b'},
 {'MCQ': 'What is the name of the architecture that LLMs are typically based on?',
  'Choices': 'a : Transformer | b : Recurrent Neural Network (RNN) | c : Long Short-Term Memory (LSTM) | d : All of the above',
  'Correct': 'a'},
 {'MCQ': 'Which model was introduced in 2017 and revolutionized language models?',
  'Choice

In [129]:
quiz = pd.DataFrame(quiz_table)

In [130]:
quiz.to_csv("Ai.csv",index=False)